<a href="https://colab.research.google.com/github/ghduf0201-oss/GPT2.0-0toHero/blob/main/notebook_06_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 6 — Toward a Tiny GPT

이제 남은 모듈들을 하나씩 추가하여 **좀 더 GPT다운 구조**로 갑니다.

이번 노트북에서 추가하는 것들:

- multi-head attention
- feedforward network
- residual connection
- layer normalization
- block stacking

In [1]:
# 1. 필수 라이브러리 강제 설치 및 임포트
!pip install -q pypdf requests

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from pypdf import PdfReader
import requests

# =====================================================================
# 🎯 [사령관 오더] 여기에 원하는 PDF 주소 링크만 입력하십시오.
# =====================================================================
pdf_url = "https://www.federalreserve.gov/mediacenter/files/FOMCpresconf20260617.pdf"
# =====================================================================

# 2. PDF 다운로드 및 텍스트 추출 공정
extracted_text = ""
if pdf_url and pdf_url.startswith("http"):
    print("🚀 Downloading PDF from URL...")
    response = requests.get(pdf_url, timeout=30)
    with open("dataset.pdf", "wb") as f:
        f.write(response.content)
    pdf_file_to_read = "dataset.pdf"
else:
    print("❌ Invalid URL.")

print("📄 Extracting text from PDF...")
reader = PdfReader(pdf_file_to_read)
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        extracted_text += page_text + "\n"

text = extracted_text

# 3. 토크나이저 구축 및 정수 텐서 변환
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

# 4. 카파시 스타일 NextTokenDataset 선언
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

# 5. 데이터 로더(DataLoader) 배치 생성 전선 가동 (사령관의 오더대로 block_size=64 세팅)
block_size = 64
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# 첫 번째 배치 샘플 수색 및 검증
xb, yb = next(iter(loader))

# 6. 최종 인프라 정산 결과 브리핑
print("\n=== 📊 Data Pipeline Process Complete ===")
print("텍스트 총 글자수 (text length) :", len(text))
print("어휘 사전 크기   (vocab_size)  :", vocab_size)
print("전체 데이터 형태 (data shape)  :", data.shape)
print("--- DataLoader Batch Check ---")
print("입력 배치 형태   (xb shape)    :", xb.shape) # 기대값: [64, 64]
print("정답 배치 형태   (yb shape)    :", yb.shape) # 기대값: [64, 64]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 19.0 MB/s eta 0:00:00
🚀 Downloading PDF from URL...
📄 Extracting text from PDF...

=== 📊 Data Pipeline Process Complete ===
텍스트 총 글자수 (text length) : 41279
어휘 사전 크기   (vocab_size)  : 77
전체 데이터 형태 (data shape)  : torch.Size([41279])
--- DataLoader Batch Check ---
입력 배치 형태   (xb shape)    : torch.Size([64, 64])
정답 배치 형태   (yb shape)    : torch.Size([64, 64])


## 1. Multi-head attention

In [2]:
class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = emb_dim // num_heads
        self.heads = nn.ModuleList([Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

## 2. Feedforward + Block

In [3]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## 3. Tiny GPT

In [4]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos
        h = self.blocks(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        return logits

model = TinyGPT(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 64, 77])


## 4. 학습

In [5]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyGPT(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.6219
epoch  1 | train loss 2.1022
epoch  2 | train loss 1.7645
epoch  3 | train loss 1.5515
epoch  4 | train loss 1.4020
epoch  5 | train loss 1.2930
epoch  6 | train loss 1.2026
epoch  7 | train loss 1.1267
epoch  8 | train loss 1.0595
epoch  9 | train loss 0.9968
epoch 10 | train loss 0.9429
epoch 11 | train loss 0.8928
epoch 12 | train loss 0.8453
epoch 13 | train loss 0.8047
epoch 14 | train loss 0.7640
epoch 15 | train loss 0.7253
epoch 16 | train loss 0.6926
epoch 17 | train loss 0.6603
epoch 18 | train loss 0.6333
epoch 19 | train loss 0.6081
epoch 20 | train loss 0.5830
epoch 21 | train loss 0.5601
epoch 22 | train loss 0.5407
epoch 23 | train loss 0.5218
epoch 24 | train loss 0.5033
epoch 25 | train loss 0.4882
epoch 26 | train loss 0.4732
epoch 27 | train loss 0.4606
epoch 28 | train loss 0.4461
epoch 29 | train loss 0.4350
epoch 30 | train loss 0.4239
epoch 31 | train loss 0.4126
epoch 32 | train loss 0.4050
epoch 33 | train loss 0.3964
epoch 34 | tra

## 5. Sampling

In [6]:
@torch.no_grad()
def sample_gpt(model, block_size, stoi, itos, device, start_text="Chairman Warsh:", max_new_tokens=400):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample_gpt(model, block_size, stoi, itos, device, start_text="Chairman Warsh:", max_new_tokens=500))

Chairman Warsh: FOMC are unambiguous and unanimous: This Committee will deliver price stability.   
At any institution, a change in leadership is a natural and timely opportunity to reaffirm 
its mission, to review current practices, and to consider whether those practices best meet our 
objectives.  My Fed colleagues and I will be worthy of a press conference.  
MICHELLE SMITH. Chris Rugaber.  
CHRIS RUGABER. Hi, Chris Rugaber at Associated Press. Thanks for taking on.  
MICHELLE SMITH. Victoria.  
VICTORIA G


## 6. 정리

이제 아주 작은 GPT 구조가 완성되었습니다.

전체 흐름은 다음과 같습니다.

1. bigram
2. MLP on names
3. MLP on Shakespeare
4. GPT-style dataset + minimal sequence model
5. single-head masked self-attention
6. tiny GPT